In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# In a notebook, we use Path.cwd() (Current Working Directory)
# This assumes your notebook is running inside your project folder.
# .parent goes up one level to your parent directory.
env_path = Path.cwd().parent / "environment.env"

print(f"Looking for .env at: {env_path}") # Debugging: Verify the path is correct

# Load the .env file
if load_dotenv(dotenv_path=env_path):
    print("Environment variables loaded successfully!")
else:
    print("Error: Could not find or load the .env file.")

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

Looking for .env at: /Users/karthik/Documents/Github/environment.env
Environment variables loaded successfully!


In [2]:

#importing necessary libraries
import os
import yaml
import logging
from dotenv import load_dotenv
import sys
# from google.colab import userdata  # Not needed since we're using env var
from ReportLabs import load_content, build_pdf  # Assuming this is available or installed in Colab

In [3]:
input_resume_path = "Resume.yaml"
with open(input_resume_path, 'r') as f:
    resume_data = yaml.safe_load(f)

In [4]:
resume_data

{'Professional Experience': {'Vestas Wind Technology': ['Achieved a 15% reduction in procurement costs by developing a data-driven supplier evaluation and conducting detailed cost analysis in supply chain operations.',
   'Streamlined purchasing processes by 20% through predictive analytics for demand forecasting, enhancing inventory management and supplier coordination.',
   'Delivered savings of $35M by developing Python-based automation tools to optimize procurement and buying workflows, improving operational efficiency.',
   'Enhanced supplier performance monitoring by implementing vendor scorecards using Excel and Power BI, providing actionable insights for procurement decisions.',
   'Integrated procurement and supply chain systems using Microsoft Power Query and SharePoint, automating operations and reducing manual efforts by 30%.',
   'Developed a comprehensive Power BI dashboard to track procurement KPIs, supplier performance, and category spend, empowering leadership with dat

In [1]:
!python ReportLabs.py 

Traceback (most recent call last):
  File "/Users/karthik/Documents/Github/Colab/ReportLabs.py", line 241, in <module>
    build_pdf(content, "Karthikeyan_Baskaran_Resume.pdf")
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/karthik/Documents/Github/Colab/ReportLabs.py", line 102, in build_pdf
    category = item.get("name", "Skills")
               ^^^^^^^^
AttributeError: 'str' object has no attribute 'get'


In [ ]:
!pip install openai

In [7]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",  # Your oMLX local address
    api_key="1234"          # Your oMLX API Key
)

response = client.chat.completions.create(
    model="gemma-4-12B-it-qat-4bit",           # Replace with your loaded model ID
    messages=[{"role": "user", "content": "If i use turbo KV cache in im macbook 24gb, with gemma 4 12b and 4 bit will it use swap?"}]
)

print(response.choices[0].message.content)


The short answer is: **It depends on your peak KV Cache size, but you are in a "danger zone" where swap is very likely to trigger.**

Here is the technical breakdown of why this happens and how to manage it on a 24GB MacBook.

### 1. The Math (Memory Budget)
To see if you'll hit swap, we have to look at how much memory is actually available to the model:

*   **Total RAM:** 24 GB.
*   **System/macOS Overhead:** Usually takes ~3–5 GB.
*   **Model Weights:** Gemma 2 9B/12B (4-bit) takes roughly **7–9 GB** of VRAM/RAM.
*   **Available for KV Cache:** This leaves you with roughly **10–12 GB** of "free" space for the context window (KV Cache) and application overhead.

### 2. How Turbo KV Cache affects this
Turbo KV Cache (or similar techniques like Paged Attention/Flash Attention) optimizes **how** the memory is managed, but it does not significantly reduce the **total amount** of memory required to store the keys and values of a long conversation.

The KV Cache grows linearly with:
1.  **